In [1]:
from sage.libs.ecl import ecl_eval
ecl_eval("(ext:set-limit 'ext:heap-size 0)")   # 0 = no ECL-side limit

<ECL: 0>

In [2]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz z')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz,
 z)

In [3]:
# e_t(e0, chi) at all PN orders (Xi^0 = paper E_0)
import re
filename = "../et_e0/NNLO_e0_faster/et_e0_NNLO_3PN.txt"
with open(filename, "r") as f:
    content = f.read()
match = re.search(r"et_e0_NNLO_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    et_e0_NNLO_3PN = (sage_eval(expr_str, locals=globals())).expand().subs({SO^2:0},{SO^3:0})

In [4]:
# k/(1+k) in x and e_t (K_files/k_xe.txt: Henry & Khalil SO + MGS nonspinning 3PN, our symbols)
filename = "K_files/k_xe.txt"
with open(filename, "r") as f:
    content = f.read()
match = re.search(r"k_xe\s*=\s*(.*)", content, re.DOTALL)
if match:
    k_xe = sage_eval(match.group(1).strip(), locals=globals())
k_e = sum(k_xe.coefficient(Xi,q).taylor(e_t,0,4)*Xi^q for q in range(2,7)).expand()

In [5]:
#a1=(et_e0_NNLO_3PN.coefficient(e0,1))
#a3=(et_e0_NNLO_3PN.coefficient(e0,3))
#a5=(et_e0_NNLO_3PN.coefficient(e0,5))
var('a1 a3 a5')
et_redefined=a1*e0+a3*e0^3

In [6]:
et_S=(e_t^2).subs({e_t:et_redefined}).taylor(e0,0,4)

In [7]:
et_SS=(e_t^4).subs({e_t:et_redefined}).taylor(e0,0,4)

In [8]:
etS_full1=et_S.coefficient(e0^4).subs({a1:(et_e0_NNLO_3PN.coefficient(e0,1))},{a3:(et_e0_NNLO_3PN.coefficient(e0,3))}).taylor(Xi,0,6)
etS_full2=et_S.coefficient(e0^2).subs({a1:(et_e0_NNLO_3PN.coefficient(e0,1))}).taylor(Xi,0,6)
etSS_full1=et_SS.coefficient(e0^4).subs({a1:(et_e0_NNLO_3PN.coefficient(e0,1))}).taylor(Xi,0,6)

et_S_final=(etS_full1)*e0^4+(etS_full2)*e0^2
et_SS_final=(etSS_full1)*e0^4

In [9]:
#e0^4 part of k
ke_eS=((k_e.collect(e_t).coefficient(e_t^2)*et_S_final).taylor(Xi,0,6))
ke_eSS=((k_e.collect(e_t).coefficient(e_t^4)*et_SS_final).taylor(Xi,0,6))
ke_final=k_e.collect(e_t).coefficient(e_t,0)+ke_eS+ke_eSS

In [10]:

k_term=ke_final

In [11]:
filename = "../phase/K_files/k_term.txt"


with open(filename, "w") as f:
    f.write("# Expansion of k_term\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("k_term =\n")
    f.write(str(k_term))
    f.write("\n\n")